In [ ]:
pip install requests python-dotenv

In [ ]:
import os, json, requests
from typing import Optional, Dict, Any, Iterable, Tuple

# Optional .env
try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

# ---------- RPC resolution ----------
def _resolve_rpc(rpc: Optional[str], access_key: Optional[str], host: str = "mainnet.gateway.tenderly.co") -> str:
    """
    Resolve Tenderly Node RPC.
    - If `rpc` is provided, use it as-is.
    - Else use TENDERLY_NODE_RPC if set.
    - Else build from TENDERLY_NODE_ACCESS_KEY and host.
      (If the env "key" already looks like a URL, use it directly.)
    """
    if rpc:
        return rpc
    env_rpc = os.getenv("TENDERLY_NODE_RPC")
    if env_rpc:
        return env_rpc
    key = access_key or os.getenv("TENDERLY_NODE_ACCESS_KEY")
    if not key:
        raise ValueError("Provide rpc= or set TENDERLY_NODE_RPC or TENDERLY_NODE_ACCESS_KEY")
    if "://" in key:
        return key
    return f"https://{host}/{key}"

# ---------- JSON-RPC helpers ----------
def _rpc(url: str, method: str, params: list, timeout: int = 180) -> Dict[str, Any]:
    payload = {"jsonrpc": "2.0", "id": 1, "method": method, "params": params}
    r = requests.post(url, json=payload, timeout=timeout)
    if not r.ok:
        raise RuntimeError(f"HTTP {r.status_code} from {url}\nBody:\n{r.text}")
    data = r.json()
    if "error" in data:
        raise RuntimeError(f"RPC error {data['error'].get('code')}: {data['error'].get('message')}")
    return data["result"]

def eth_get_tx_by_hash(tx_hash: str, rpc: Optional[str] = None, access_key: Optional[str] = None) -> Dict[str, Any]:
    url = _resolve_rpc(rpc, access_key)
    return _rpc(url, "eth_getTransactionByHash", [tx_hash])

def tenderly_simulate_transaction(
    call_obj: Dict[str, Any],
    block: str = "latest",                    # can be hex block (e.g., '0x1234') or tags: latest|finalized|safe|pending|earliest
    state_overrides: Optional[Dict[str, Any]] = None,
    block_overrides: Optional[Dict[str, Any]] = None,
    rpc: Optional[str] = None,
    access_key: Optional[str] = None,
    timeout: int = 180,
) -> Tuple[Dict[str, Any], str]:
    """
    Call tenderly_simulateTransaction and return (result, used_url).
    `call_obj` keys can include: from, to, gas, gasPrice, value, input, nonce,
    maxFeePerGas, maxPriorityFeePerGas, accessList, maxFeePerBlobGas, blobVersionedHashes.
    """
    url = _resolve_rpc(rpc, access_key)
    params = [call_obj, block]
    # Only append overrides if provided (per method signature)
    if state_overrides is not None:
        params.append(state_overrides)
        if block_overrides is not None:
            params.append(block_overrides)
    elif block_overrides is not None:
        # If you only want block overrides, you must pass an empty {} for state overrides
        params.append({})
        params.append(block_overrides)

    result = _rpc(url, "tenderly_simulateTransaction", params)
    return result, url

# ---------- Converters ----------
def build_call_from_tx(tx: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convert eth_getTransactionByHash result into a call object suitable for simulation.
    We copy only relevant fields that exist.
    """
    out = {}
    for k_src, k_dst in [
        ("from", "from"),
        ("to", "to"),
        ("gas", "gas"),
        ("gasPrice", "gasPrice"),
        ("value", "value"),
        ("input", "input"),
        ("nonce", "nonce"),
        ("maxFeePerGas", "maxFeePerGas"),
        ("maxPriorityFeePerGas", "maxPriorityFeePerGas"),
        ("accessList", "accessList"),
        ("maxFeePerBlobGas", "maxFeePerBlobGas"),
        ("blobVersionedHashes", "blobVersionedHashes"),
    ]:
        v = tx.get(k_src)
        if v is not None:
            out[k_dst] = v
    # If no gas provided (rare for historical tx objects), you can estimate or set a cap.
    if "gas" not in out:
        out["gas"] = hex(30_000_000)  # generous default; adjust as needed
    return out

# ---------- Light inspection ----------
def _walk(obj: Any, path: Tuple = ()) -> Iterable[Tuple[Tuple, Any]]:
    if isinstance(obj, dict):
        yield path, obj
        for k, v in obj.items():
            yield from _walk(v, path + (k,))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            yield from _walk(v, path + (i,))

def summarize_simulation(sim_result: Dict[str, Any], limit: int = 12):
    """
    Heuristically surface revert-ish nodes in the decoded trace.
    """
    hits = []
    for path, node in _walk(sim_result):
        if isinstance(node, dict):
            keys = set(k.lower() for k in node.keys())
            if any(k in keys for k in ("error", "revertreason", "reason", "reverted")):
                ctx = {
                    "path": path,
                    "functionName": node.get("functionName"),
                    "type": node.get("type"),
                    "from": node.get("from"),
                    "to": node.get("to"),
                    "reverted": node.get("reverted"),
                    "error": node.get("error") or node.get("reason") or node.get("revertReason"),
                }
                hits.append(ctx)
                if len(hits) >= limit:
                    break
    if not hits:
        print("No obvious revert markers found. Inspect the saved JSON; schema can vary.")
        return
    print(f"Found {len(hits)} potential revert frames (showing up to {limit}):")
    for i, h in enumerate(hits, 1):
        print(f"\n[{i}] path={h['path']}")
        for k in ("functionName","type","from","to","reverted","error"):
            if h.get(k) is not None:
                print(f"  {k}: {h[k]}")

def pretty_dump(obj: Any, *, path: Optional[str] = None, preview: bool = False, preview_chars: int = 7000):
    s = json.dumps(obj, indent=2)
    print(s[:preview_chars] + ("..." if preview and len(s) > preview_chars else ""))
    if path:
        with open(path, "w") as f:
            f.write(s)
        print(f"\nSaved: {path}")


In [ ]:
# Your reverted tx hash
TX = "0x47331fd359673f43fcba62b9b7f88527b78fdd82d8815ce8716a4a0b2b3e7e45"

# 1) Fetch the original tx
tx_obj = eth_get_tx_by_hash(TX)
print("Fetched tx:")
pretty_dump(tx_obj, preview=True, preview_chars=1200)

# 2) Build call object from the original tx
call = build_call_from_tx(tx_obj)

# 3) Choose the block to reproduce original state (use the original tx's blockNumber if available)
block = tx_obj.get("blockNumber") or "latest"

# (Optional) State/Block overrides — leave None for a faithful replay
state_overrides = None
block_overrides = None
# Example override shape (leave commented unless you really need this):
# state_overrides = {
#   "0xYourAddress": {
#       "balance": "0x3635c9adc5dea00000",  # 1000 ETH
#       "nonce": "0x1e",
#       "stateDiff": {"0x<slot>": "0x<value>"},
#       "code": "0x60016000..."  # bytecode override
#   }
# }
# block_overrides = {"time": hex(1734444444), "gasLimit": hex(30_000_000), "baseFee": hex(1_000_000_000)}

# 4) Run the simulation
sim_result, used_url = tenderly_simulate_transaction(
    call_obj=call,
    block=block,
    state_overrides=state_overrides,
    block_overrides=block_overrides,
)
print("Using RPC:", used_url)

# 5) Save & preview
pretty_dump(sim_result, path="sim_tenderly.json", preview=True, preview_chars=8000)

# 6) Surface revert-ish frames for quick inspection
summarize_simulation(sim_result)


## Let's use the simulator